

# Лабораторная работа 6


In [ ]:
import pandas as pd
import numpy as np

order_items = pd.read_csv("/content/olist_order_items_dataset.csv")
orders = pd.read_csv('/content/olist_orders_dataset.csv')
customers = pd.read_csv('/content/olist_customers_dataset.csv')
products = pd.read_csv('/content/olist_products_dataset.csv')

# Задание 1: Оптимизация памяти (Categorical Data)

In [ ]:
#Измерение: Оцените текущее потребление памяти вашим
#объединенным датасетом с помощью df.info(memory_usage='deep').

df = pd.merge(orders, customers, on='customer_id', how='left')
df = pd.merge(df, order_items, on='order_id', how='left')
df = pd.merge(df, products, on='product_id', how='left')

df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       113425 non-null  object 
 1   customer_id                    113425 non-null  object 
 2   order_status                   113425 non-null  object 
 3   order_purchase_timestamp       113425 non-null  object 
 4   order_approved_at              113264 non-null  object 
 5   order_delivered_carrier_date   111457 non-null  object 
 6   order_delivered_customer_date  110196 non-null  object 
 7   order_estimated_delivery_date  113425 non-null  object 
 8   customer_unique_id             113425 non-null  object 
 9   customer_zip_code_prefix       113425 non-null  int64  
 10  customer_city                  113425 non-null  object 
 11  customer_state                 113425 non-null  object 
 12  order_item_id                 

memory usage: 122.2 MB

In [ ]:
# Трансформация: Переведите колонки customer_state, order_status и
# product_category_name в тип category.

df['customer_state'] = df['customer_state'].astype('category')
df['order_status'] = df['order_status'].astype('category')
df['product_category_name'] = df['product_category_name'].astype('category')

df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype   
---  ------                         --------------   -----   
 0   order_id                       113425 non-null  object  
 1   customer_id                    113425 non-null  object  
 2   order_status                   113425 non-null  category
 3   order_purchase_timestamp       113425 non-null  object  
 4   order_approved_at              113264 non-null  object  
 5   order_delivered_carrier_date   111457 non-null  object  
 6   order_delivered_customer_date  110196 non-null  object  
 7   order_estimated_delivery_date  113425 non-null  object  
 8   customer_unique_id             113425 non-null  object  
 9   customer_zip_code_prefix       113425 non-null  int64   
 10  customer_city                  113425 non-null  object  
 11  customer_state                 113425 non-null  category
 12  order_item_id   

Сравнение: Замерьте объем памяти «до» и «после».

До: 122.2 MB

После: 103.9 MB

Экономия оказалась ниже ожидаемых 50–80%, тк основную часть памяти в датасете занимают числовые столбцы и другие типы данных, а не текстовые категориальные признаки.

# Задание 2: Автоматизация подготовки (Sklearn Pipelines)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
# Создайте ColumnTransformer, который включает:
#   Для численных признаков ( price, freight_value):
#     заполнение пропусков медианой и масштабирование ( StandardScaler).
#   Для категориальных признаков: One-Hot Encoding или Ordinal Encoding.

nums = ['price', 'freight_value']
cats = ['customer_state', 'order_status', 'product_category_name']

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, nums),
    ('cat', cat_transformer, cats)
])

# Оберните это в Pipeline и протестируйте его на небольшом фрагменте данных.

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

sample = df.head(5)

result = pipeline.fit_transform(sample)

print(result)
print(result.shape)
preprocessor

[[-0.8141075  -1.15235788  0.          0.          0.          1.
   1.          0.          0.          0.          0.          1.        ]
 [ 0.80125164  0.72805875  1.          0.          0.          0.
   1.          0.          0.          1.          0.          0.        ]
 [ 1.55148043  0.2539366   0.          1.          0.          0.
   1.          1.          0.          0.          0.          0.        ]
 [-0.54078385  1.32272041  0.          0.          1.          0.
   1.          0.          0.          0.          1.          0.        ]
 [-0.99784072 -1.15235788  0.          0.          0.          1.
   1.          0.          1.          0.          0.          0.        ]]
(5, 12)


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['price', 'freight_value']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['customer_state', 'order_status',
                                  'product_category_name'])])

# Задание 3: Высокопроизводительные запросы (pd.query)


In [ ]:
# Реализуйте сложный фильтр (например:
# "заказы из штатов SP или RJ со стоимостью выше 150 BRL,
# сделанные в 2018 году") двумя способами:

# 1. Через классические булевы маски.
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

filtered_mask = df[(
        (df['customer_state'].isin(['SP', 'RJ'])) &
        (df['price'] > 150) &
        (df['order_purchase_timestamp'].dt.year == 2018)
    )
]

# 2.Через метод .query().
df['purchase_year'] = df['order_purchase_timestamp'].dt.year
filtered_query = df.query("customer_state in ['SP', 'RJ'] and price > 150 and purchase_year == 2018")

# Сравнение

%timeit df[((df['customer_state'].isin(['SP', 'RJ'])) & (df['price'] > 150) & (df['purchase_year'] == 2018))]

%timeit df.query("customer_state in ['SP', 'RJ'] and price > 150 and purchase_year == 2018")

5.33 ms ± 746 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
11.2 ms ± 381 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Фильтрация данных была реализована двумя способами:

* через булевы маски
* через метод pd.query()

Метод .query() показал более читаемый результат на текущем объеме данных. А булевые маски более производительные.

In [ ]:
!pip install pyarrow

In [ ]:
# Сохраните финальный датасет в формате HDF5 (или Parquet).
df.to_parquet('orders.parquet', index=False)
df_parquet = pd.read_parquet('orders.parquet')
# Продемонстрируйте разницу: сравните размер файла на диске и
# скорость загрузки обратно в Pandas по сравнению с исходным CSV.

import os

csv_size = os.path.getsize('olist_orders_dataset.csv') / 1024**2
parquet_size = os.path.getsize('orders.parquet') / 1024**2

print(f'CSV: {csv_size:.2f} MB')
print(f'Parquet: {parquet_size:.2f} MB')

%timeit pd.read_csv('olist_orders_dataset.csv')
%timeit pd.read_parquet('orders.parquet')

CSV: 16.84 MB
Parquet: 18.52 MB
488 ms ± 55.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
248 ms ± 13.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Размер CSV-файла 16.84 MB, а Parquet — 18.52 MB. Несмотря на немного больший размер файла, формат Parquet показал значительно более высокую скорость загрузки данных.

Время чтения CSV: 488 ms ± 55.2 ms

Время чтения Parquet: 248 ms ± 13.4 ms

Таким образом, загрузка Parquet выполняется примерно в 2 раза быстрее.

Кроме ускорения чтения, формат Parquet сохраняет типы данных (datetime, category, числовые типы), что особенно важно для аналитики и пайплайнов. В отличие от CSV, при повторной загрузке не требуется дополнительное преобразование типов данных.